In [ ]:
# imports 
import ast
import difflib
import hashlib
import json
import math
import os
import re
import unicodedata
from pathlib import Path
from typing import Annotated
from urllib.parse import quote_plus
from difflib import SequenceMatcher
from functools import lru_cache

import numexpr
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

import ifcopenshell

In [ ]:
def load_llm(id_model, temperature):
    llm = ChatOpenAI(
        model=id_model,
        temperature=temperature,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
    return llm

In [ ]:
openai_api_key = os.environ.get("OPENAI_API_KEY") or os.environ.get("openai_api_key")
if not openai_api_key:
    raise EnvironmentError("Defina OPENAI_API_KEY ou openai_api_key antes de criar o LLM.")
os.environ["OPENAI_API_KEY"] = openai_api_key
id_model = "gpt-5.5"
temperature = 0.3

llm = load_llm(id_model, temperature)

In [ ]:
# Math tools
@tool
def calculator_tool(expression: str) -> str:
    """Use para qualquer cálculo aritmético direto; não faça cálculo numérico de cabeça."""
    local_dict = {"pi": math.pi, "e": math.e}
    return str(
        numexpr.evaluate(
            expression.strip(),
            global_dict={},
            local_dict=local_dict,
        )
    )


@tool
def percentage_tool(base_value: float, percent: float) -> dict:
    """Use para calcular porcentagem aplicada sobre um valor base; não faça cálculo de cabeça."""
    percentage_value = base_value * (percent / 100)
    total_with_percentage = base_value + percentage_value
    return {
        "base_value": base_value,
        "percent": percent,
        "percentage_value": percentage_value,
        "total_with_percentage": total_with_percentage,
    }


@tool
def percent_change_tool(initial_value: float, final_value: float) -> dict:
    """Use para calcular variação percentual entre dois valores; não faça cálculo de cabeça."""
    if initial_value == 0:
        return {
            "error": "initial_value não pode ser zero para calcular variação percentual."
        }

    absolute_change = final_value - initial_value
    percent_change = (absolute_change / initial_value) * 100
    return {
        "initial_value": initial_value,
        "final_value": final_value,
        "absolute_change": absolute_change,
        "percent_change": percent_change,
    }

In [ ]:
# Lendo ifc com o ifcOpenShell
file_path = "C:\\Users\\Usuario2026\\workspaces\\projeto Obra Barata\\exemplos\\planta2\\teste2.ifc"
ifc = ifcopenshell.open(file_path)  # Substitua pelo caminho do seu arquivo IFC

In [ ]:
def _entity_id(entity):
    try:
        return entity.id()
    except Exception:
        return None


def _entity_name(entity):
    if entity is None:
        return None
    for attr in ("Name", "LongName", "Description"):
        value = getattr(entity, attr, None)
        if value:
            return str(value)
    try:
        return f"{entity.is_a()} #{entity.id()}"
    except Exception:
        return str(entity)


def _entity_is_a(entity, type_name):
    try:
        return entity.is_a(type_name)
    except Exception:
        return False


def _layers_from_material_select(material_select):
    if material_select is None:
        return None, []

    layer_set = material_select
    if _entity_is_a(material_select, "IfcMaterialLayerSetUsage"):
        layer_set = getattr(material_select, "ForLayerSet", None)

    if _entity_is_a(layer_set, "IfcMaterialLayerSet"):
        return layer_set, list(getattr(layer_set, "MaterialLayers", []) or [])

    if _entity_is_a(material_select, "IfcMaterialLayer"):
        return None, [material_select]

    return None, []


def _layer_to_dict(layer):
    material = getattr(layer, "Material", None)
    return {
        "id": _entity_id(layer),
        "nome": _entity_name(layer),
        "material": _entity_name(material),
        "material_id": _entity_id(material),
        "espessura": getattr(layer, "LayerThickness", None),
        "categoria": getattr(layer, "Category", None),
        "prioridade": getattr(layer, "Priority", None),
        "ventilada": getattr(layer, "IsVentilated", None),
    }


def _usage_to_dict(material_select):
    if not _entity_is_a(material_select, "IfcMaterialLayerSetUsage"):
        return None
    return {
        "id": _entity_id(material_select),
        "layer_set_direction": getattr(material_select, "LayerSetDirection", None),
        "direction_sense": getattr(material_select, "DirectionSense", None),
        "offset_from_reference_line": getattr(material_select, "OffsetFromReferenceLine", None),
    }


def _related_object_to_dict(obj):
    return {
        "id": _entity_id(obj),
        "global_id": getattr(obj, "GlobalId", None),
        "tipo": obj.is_a() if hasattr(obj, "is_a") else None,
        "nome": _entity_name(obj),
    }


def extrair_layersets(ifc_file):
    """Extrai layer sets de material do contexto IFC em formato serializavel."""
    layer_sets = {}

    def add_layer_set(material_select, related_objects=None):
        layer_set, layers = _layers_from_material_select(material_select)
        if not layers:
            return

        layers_data = [_layer_to_dict(layer) for layer in layers]
        key_entity = layer_set or material_select
        key = _entity_id(key_entity) or f"anon-{len(layer_sets) + 1}"
        if key not in layer_sets:
            layer_sets[key] = {
                "id": _entity_id(layer_set),
                "nome": _entity_name(layer_set) or _entity_name(material_select),
                "tipo_origem": material_select.is_a() if hasattr(material_select, "is_a") else None,
                "uso": _usage_to_dict(material_select),
                "camadas": layers_data,
                "espessura_total": sum(layer.get("espessura") or 0 for layer in layers_data),
                "aplicado_em": [],
            }

        if related_objects:
            existing = {
                item.get("global_id") or item.get("id")
                for item in layer_sets[key]["aplicado_em"]
            }
            for obj in related_objects:
                obj_data = _related_object_to_dict(obj)
                obj_key = obj_data.get("global_id") or obj_data.get("id")
                if obj_key not in existing:
                    layer_sets[key]["aplicado_em"].append(obj_data)
                    existing.add(obj_key)

    for rel in ifc_file.by_type("IfcRelAssociatesMaterial"):
        add_layer_set(
            getattr(rel, "RelatingMaterial", None),
            getattr(rel, "RelatedObjects", None),
        )

    try:
        for layer_set in ifc_file.by_type("IfcMaterialLayerSet"):
            add_layer_set(layer_set)
    except Exception:
        pass

    return sorted(
        layer_sets.values(),
        key=lambda item: (item.get("nome") or "", item.get("id") or 0),
    )


In [ ]:
# Safe digest builder for IFC2X3 / IFC4
ENTITY_ALIASES = {
    "IfcPipeSegment": ["IfcPipeSegment", "IfcFlowSegment"],
    "IfcPipeFitting": ["IfcPipeFitting", "IfcFlowFitting"],
    "IfcFlowTerminal": ["IfcFlowTerminal"],
    "IfcSwitchingDevice": ["IfcSwitchingDevice", "IfcSwitchDevice"],
}

def _entity_exists(schema_name: str, entity_name: str) -> bool:
    try:
        schema = ifcopenshell.ifcopenshell_wrapper.schema_by_name(schema_name)
        return schema.declaration_by_name(entity_name) is not None
    except Exception:
        return False

def _safe_count(m, *entity_names):
    total = 0
    for entity_name in entity_names:
        try:
            if _entity_exists(m.schema, entity_name):
                total += len(m.by_type(entity_name))
        except Exception:
            pass
    return total

def safe_build_digest(ifc_file):
    m = ifc_file

    def resolve_type(type_name):
        aliases = ENTITY_ALIASES.get(type_name, [type_name])
        for alias in aliases:
            if _entity_exists(m.schema, alias):
                return alias
        return None

    areas = {
        "fundacao":       {"tipos": ["IfcFooting", "IfcPile"]},
        "estrutura":      {"tipos": ["IfcColumn", "IfcBeam"]},
        "alvenaria":      {"tipos": ["IfcWall", "IfcWallStandardCase"]},
        "cobertura":      {"tipos": ["IfcRoof"]},
        "portas_janelas": {"tipos": ["IfcDoor", "IfcWindow"]},
        "hidraulicas":    {"tipos": ["IfcPipeSegment", "IfcPipeFitting", "IfcFlowTerminal"]},
        "eletricas":      {"tipos": ["IfcLightFixture", "IfcOutlet", "IfcSwitchingDevice", "IfcCableCarrierSegment"]},
        "revestimentos":  {"tipos": ["IfcCovering"]},
        "loucas_metais":  {"tipos": ["IfcSanitaryTerminal"]},
    }

    for area in areas.values():
        resolved = []
        for t in area["tipos"]:
            resolved_name = resolve_type(t)
            if resolved_name:
                resolved.append(resolved_name)

        area["presente"] = _safe_count(m, *resolved) > 0
        area["n"] = _safe_count(m, *resolved)

    return {
        "schema": m.schema,
        "pavimentos": [s.Name for s in m.by_type("IfcBuildingStorey")],
        "areas": areas,
        "materiais": sorted({x.Name for x in m.by_type("IfcMaterial")}),
        "camadas_material": extrair_layersets(m),
    }

# run the safe version
build_digest_result = safe_build_digest(ifc)
build_digest_result

In [ ]:
# Models
import importlib
import sys
from pathlib import Path

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "backend/api/src/app/models/materials.py").exists():
    project_root = project_root.parent

api_src_path = project_root / "backend/api/src"
if str(api_src_path) not in sys.path:
    sys.path.insert(0, str(api_src_path))

from app.models import materials as material_models

material_models = importlib.reload(material_models)
AreaMateriaisObra = material_models.AreaMateriaisObra
AreaObra = material_models.AreaObra
ListaMateriaisObra = material_models.ListaMateriaisObra
MaterialObra = material_models.MaterialObra
OfertaFornecedor = material_models.OfertaFornecedor
OrigemMaterial = material_models.OrigemMaterial
PerfilProduto = material_models.PerfilProduto

schema = ListaMateriaisObra.model_json_schema()
assert schema.get("additionalProperties") is False, schema


In [ ]:
# Validação do JSON base de materiais
materials_template_path = Path("backend/ml/prompts/IFC_construction_details.json")
if not materials_template_path.exists():
    materials_template_path = Path("prompts/IFC_construction_details.json")

with materials_template_path.open(encoding="utf-8") as file:
    materials_payload = json.load(file)

lista_materiais_base = ListaMateriaisObra.model_validate(materials_payload)
lista_materiais_base


In [ ]:
# LLM: digest IFC -> ListaMateriaisObra
def gerar_lista_materiais_da_obra(
    build_digest_result: dict,
    lista_base: ListaMateriaisObra,
    llm_model=None,
) -> ListaMateriaisObra:
    """Preenche a lista de materiais com base no digest IFC e inferencias de obra."""
    llm_model = llm_model or llm
    schema = ListaMateriaisObra.model_json_schema()
    if schema.get("additionalProperties") is not False:
        raise ValueError("ListaMateriaisObra precisa ter additionalProperties=false no schema.")

    try:
        structured_llm = llm_model.with_structured_output(
            ListaMateriaisObra,
            method="json_schema",
            strict=True,
        )
    except TypeError:
        structured_llm = llm_model.with_structured_output(ListaMateriaisObra)

    system_prompt = """
Voce e um especialista em planejamento de compras para obras residenciais brasileiras.
Sua tarefa e transformar um digest tecnico de um arquivo IFC em uma ListaMateriaisObra.

Use o build_digest_result como fonte principal do que existe no projeto. Ele pode conter:
- schema IFC;
- pavimentos;
- areas detectadas e contagens de entidades IFC;
- materiais extraidos diretamente do IFC;
- camadas de materiais de paredes/coberturas/revestimentos.

Use a lista_base como catalogo inicial de materiais e formato esperado, mas nao copie tudo cegamente.
Inclua materiais quando houver evidencia direta no IFC ou quando forem consumiveis/insumos
necessarios para executar elementos detectados.

Regras de preenchimento:
1. Retorne apenas dados no schema ListaMateriaisObra.
2. Organize por area de compra: Fundacao, Estrutura, Alvenaria, Cobertura,
   Esquadrias, Portas e janelas, Instalacoes hidraulicas, Instalacoes eletricas,
   Revestimentos, Loucas e metais, Pintura, Impermeabilizacao e demais categorias aplicaveis.
3. Para materiais diretamente citados em build_digest_result.materiais ou camadas_material,
   use origem='ifc', nivel_confianca entre 75 e 95 e referencias_ifc com ids, nomes
   de materiais, layers ou objetos relacionados quando disponiveis.
4. Para materiais nao modelados mas implicitamente necessarios, use origem='ia',
   nivel_confianca entre 45 e 75 e escreva justificativa clara.
5. Quantidade deve ficar null quando o digest nao trouxer base suficiente para calcular.
   Nao invente metragem, volume ou unidade contavel sem evidencia.
6. Precos, fornecedor, frete, parcelas, valor_unitario, valor_total,
   preco_a_vista e preco_a_prazo devem ficar vazios/null nesta etapa.
   lista_fornecedores deve ficar como lista vazia [] nesta etapa.
7. Use perfil_produto='Medio custo' como padrao quando o item for comprado em loja.
8. Nao duplique o mesmo material dentro da mesma area. Consolide itens equivalentes.

Inferencias obrigatorias quando houver evidencia contextual:
- Se houver banheiro, loucas/metais, IfcSanitaryTerminal, ambiente com nome banheiro,
  lavabo ou suite, inclua itens como vaso sanitario, assento sanitario, lavatorio/cuba,
  torneira, sifao, engate flexivel, ralo, registro e chuveiro/ducha quando aplicavel.
- Se houver cozinha, copa, area gourmet, ponto hidraulico de cozinha ou evidencia de pia,
  inclua pia/cuba de cozinha, torneira de cozinha, sifao, engate flexivel, bancada ou apoio
  quando coerente com o projeto.
- Se houver portas e janelas via IfcDoor/IfcWindow, inclua folha ou esquadria, marco/batente,
  fechadura, dobradicas, guarnicao, vidro, silicone/vedacao e fixadores conforme aplicavel.
- Se houver paredes/alvenaria/camadas de bloco, inclua bloco/tijolo, argamassa de assentamento,
  cimento, areia, cal, vergas/contravergas e perdas tecnicas quando coerente.
- Se houver revestimentos ou areas molhadas, inclua argamassa colante, rejunte,
  impermeabilizante, porcelanato/ceramica ou azulejo quando houver evidencia de acabamento.
- Se houver cobertura, inclua telhas, madeira/estrutura auxiliar, manta/subcobertura,
  cumeeira, parafusos/pregos/fixadores e impermeabilizacao quando coerente.
- Se houver instalacoes eletricas, inclua eletrodutos, cabos, caixas, tomadas,
  interruptores, disjuntores e quadro de distribuicao, mantendo quantidades null sem base.
- Se houver instalacoes hidraulicas, inclua tubos, conexoes, registros, caixa d'agua,
  ralos e materiais de vedacao, mantendo quantidades null sem base.
- Se houver pintura ou camadas que indiquem acabamento de parede, inclua selador,
  massa corrida/acrilica, tinta e lixas.

Se uma inferencia for fraca, ainda pode incluir o item, mas reduza o nivel_confianca
e explique a incerteza na justificativa. A resposta final sera revisada por humano.
""".strip()

    user_payload = {
        "build_digest_result": build_digest_result,
        "lista_base": lista_base.model_dump(mode="json"),
    }

    response = structured_llm.invoke(
        [
            ("system", system_prompt),
            (
                "user",
                "Preencha a ListaMateriaisObra a partir deste JSON:\n"
                + json.dumps(user_payload, ensure_ascii=False, indent=2),
            ),
        ]
    )
    return response


lista_materiais_obra = gerar_lista_materiais_da_obra(
    build_digest_result=build_digest_result,
    lista_base=lista_materiais_base,
)
lista_materiais_obra


In [23]:
# Dados espaciais e quantitativos do IFC
def _safe_by_type(ifc_file, entity_name):
    try:
        return list(ifc_file.by_type(entity_name))
    except Exception:
        return []


def _safe_value(value):
    if hasattr(value, "wrappedValue"):
        return value.wrappedValue
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)


def _element_ref(element):
    return {
        "id": _entity_id(element),
        "global_id": getattr(element, "GlobalId", None),
        "tipo": element.is_a() if hasattr(element, "is_a") else None,
        "nome": _entity_name(element),
    }


def _extract_properties_and_quantities(element):
    data = {"property_sets": {}, "quantities": {}}
    for rel in getattr(element, "IsDefinedBy", []) or []:
        definition = getattr(rel, "RelatingPropertyDefinition", None)
        if definition is None:
            continue

        name = getattr(definition, "Name", None) or definition.is_a()
        if _entity_is_a(definition, "IfcPropertySet"):
            props = {}
            for prop in getattr(definition, "HasProperties", []) or []:
                prop_name = getattr(prop, "Name", None)
                if not prop_name:
                    continue
                value = getattr(prop, "NominalValue", None)
                props[prop_name] = _safe_value(value)
            if props:
                data["property_sets"][name] = props

        if _entity_is_a(definition, "IfcElementQuantity"):
            qtos = {}
            for quantity in getattr(definition, "Quantities", []) or []:
                qto_name = getattr(quantity, "Name", None)
                if not qto_name:
                    continue
                for attr in (
                    "AreaValue",
                    "VolumeValue",
                    "LengthValue",
                    "CountValue",
                    "WeightValue",
                    "TimeValue",
                ):
                    value = getattr(quantity, attr, None)
                    if value is not None:
                        qtos[qto_name] = {"tipo": attr, "valor": _safe_value(value)}
                        break
            if qtos:
                data["quantities"][name] = qtos
    return data


def _spatial_container_name(element):
    for rel in getattr(element, "ContainedInStructure", []) or []:
        container = getattr(rel, "RelatingStructure", None)
        if container is not None:
            return _entity_name(container)
    return None


def extrair_dados_espaciais(ifc_file):
    """Extrai ambientes, pavimentos, contenção espacial e quantidades IFC."""
    storeys = []
    for storey in _safe_by_type(ifc_file, "IfcBuildingStorey"):
        contained = []
        for rel in getattr(storey, "ContainsElements", []) or []:
            contained.extend(getattr(rel, "RelatedElements", []) or [])
        counts = {}
        for element in contained:
            element_type = element.is_a() if hasattr(element, "is_a") else "Unknown"
            counts[element_type] = counts.get(element_type, 0) + 1
        storeys.append({
            "id": _entity_id(storey),
            "global_id": getattr(storey, "GlobalId", None),
            "nome": _entity_name(storey),
            "elevacao": getattr(storey, "Elevation", None),
            "contagem_elementos": counts,
        })

    spaces = []
    for space in _safe_by_type(ifc_file, "IfcSpace"):
        contained = []
        for rel in getattr(space, "ContainsElements", []) or []:
            contained.extend(getattr(rel, "RelatedElements", []) or [])
        spaces.append({
            "id": _entity_id(space),
            "global_id": getattr(space, "GlobalId", None),
            "nome": _entity_name(space),
            "long_name": getattr(space, "LongName", None),
            "object_type": getattr(space, "ObjectType", None),
            "pavimento": _spatial_container_name(space),
            "dados": _extract_properties_and_quantities(space),
            "elementos_contidos": [_element_ref(element) for element in contained],
        })

    element_types = [
        "IfcWall", "IfcWallStandardCase", "IfcSlab", "IfcRoof", "IfcCovering",
        "IfcDoor", "IfcWindow", "IfcColumn", "IfcBeam", "IfcFooting", "IfcPile",
        "IfcFlowSegment", "IfcFlowFitting", "IfcFlowTerminal", "IfcSanitaryTerminal",
        "IfcLightFixture", "IfcOutlet",
    ]
    elementos = []
    for entity_name in element_types:
        for element in _safe_by_type(ifc_file, entity_name):
            elementos.append({
                **_element_ref(element),
                "pavimento_ou_ambiente": _spatial_container_name(element),
                "dados": _extract_properties_and_quantities(element),
            })

    return {
        "schema": ifc_file.schema,
        "pavimentos": storeys,
        "ambientes": spaces,
        "elementos_com_quantitativos": elementos,
    }


# Usa o extrator padronizado do backend, incluindo fallback geometrico.
from app.services.ifc.extractor import extract_spatial_data as extrair_dados_espaciais

dados_espaciais_ifc = extrair_dados_espaciais(ifc)
dados_espaciais_ifc


{'schema': 'IFC2X3',
 'pavimentos': [{'id': 123,
   'global_id': '3kJATcCWX1SOOS7vY7bhdt',
   'nome': 'Fundações',
   'elevacao': 0.0,
   'contagem_elementos': {'IfcSlab': 2,
    'IfcWall': 2,
    'IfcWallStandardCase': 2,
    'IfcStair': 1}},
  {'id': 129,
   'global_id': '3kJATcCWX1SOOS7vY7aWLd',
   'nome': 'Pavimento Térreo',
   'elevacao': 1.0,
   'contagem_elementos': {'IfcWall': 8,
    'IfcWallStandardCase': 3,
    'IfcSlab': 2,
    'IfcCovering': 3,
    'IfcDoor': 5,
    'IfcBuildingElementProxy': 1,
    'IfcWindow': 5,
    'IfcFlowTerminal': 5}},
  {'id': 135,
   'global_id': '3kJATcCWX1SOOS7vY7bhfs',
   'nome': 'Cobertura',
   'elevacao': 3.7,
   'contagem_elementos': {'IfcRoof': 1, 'IfcWallStandardCase': 2}}],
 'ambientes': [],
 'elementos_com_quantitativos': [{'id': 1004,
   'global_id': '21SiID7FH5XQIgWZAEgVi9',
   'tipo': 'IfcWall',
   'nome': 'Parede básica:EXT.14-ALV:312064',
   'pavimento_ou_ambiente': 'Pavimento Térreo',
   'dados': {'property_sets': {'Pset_QuantityTak

In [ ]:
# LLM: ListaMateriaisObra + dados espaciais IFC -> ListaMateriaisObra com quantidades
def estimar_quantidades_materiais(
    lista_materiais: ListaMateriaisObra,
    dados_espaciais_ifc: dict,
    build_digest_result: dict | None = None,
    llm_model=None,
) -> ListaMateriaisObra:
    """Estima quantidades a partir de areas, volumes, comprimentos e contagens IFC."""
    llm_model = llm_model or llm
    schema = ListaMateriaisObra.model_json_schema()
    if schema.get("additionalProperties") is not False:
        raise ValueError("ListaMateriaisObra precisa ter additionalProperties=false no schema.")

    try:
        structured_llm = llm_model.with_structured_output(
            ListaMateriaisObra,
            method="json_schema",
            strict=True,
        )
    except TypeError:
        structured_llm = llm_model.with_structured_output(ListaMateriaisObra)

    system_prompt = """
Voce é um orcamentista BIM para obras residenciais brasileiras.
Recebera uma ListaMateriaisObra ja consolidada e dados espaciais/quantitativos extraidos do IFC.
Sua tarefa e devolver a mesma ListaMateriaisObra preenchendo o campo quantidade de cada material
quando houver base tecnica suficiente nos dados espaciais, nos quantitativos IFC ou nas contagens.

Regras obrigatorias:
1. Retorne apenas o schema ListaMateriaisObra.
2. Preserve todas as areas e materiais recebidos. Nao remova materiais.
3. Pode ajustar medida quando necessario para combinar com a quantidade calculada.
4. Preencha quantidade quando houver area, volume, comprimento, contagem coerente
5. Se nao houver base numerica suficiente, estime o quanto seria gasto, por ex, quantidade de sacos de cimento para argamassa de assentamento, e explique o cálculo feito.
6. Mantenha fornecedor, lista_fornecedores, valores, frete e parcelas vazios/null.
7. Atualize nivel_confianca: 80-95 para quantitativo direto do IFC; 60-79 para conversao tecnica
   com regra clara; 40-59 para estimativa fraca que ainda exige revisao.
8. Em referencias_ifc, cite o ambiente, elemento, Qto, material layer ou contagem usada.

Diretrizes de quantificacao:
- Materiais em m2: use NetArea, GrossArea, AreaValue ou area de ambientes/superficies.
- Materiais em m3: use NetVolume, GrossVolume, VolumeValue ou volume de elementos.
- Materiais em metro linear: use LengthValue, comprimento de tubos, vigas, rodapes ou perfis.
- Materiais em unidades: use contagem direta de IfcDoor, IfcWindow, IfcSanitaryTerminal,
  IfcOutlet, IfcLightFixture ou elementos equivalentes.
- Portas e janelas: conte IfcDoor/IfcWindow; ferragens podem seguir a contagem de portas,
  mas explique a regra usada.
- Loucas e metais: use IfcSanitaryTerminal ou contagem clara de ambientes banheiro/lavabo/suite.
- Revestimentos: use area de ambientes molhados, IfcCovering ou camadas de material quando houver area.
- Pintura: use area de paredes/forros quando houver; caso so exista material layer sem area,
  mantenha quantidade null.
- Argamassas, rejuntes, tintas e impermeabilizantes: so converta para sacos/latas/baldes se houver
  area/volume base e uma taxa de consumo explicitada na justificativa. Se a taxa for uma regra usual,
  marque origem='ia' ou reduza a confianca.

A saida sera revisada por humano, entao prefira ser conservador a preencher quantidade sem base.
""".strip()

    user_payload = {
        "lista_materiais": lista_materiais.model_dump(mode="json"),
        "dados_espaciais_ifc": dados_espaciais_ifc,
        "build_digest_result": build_digest_result or {},
    }

    return structured_llm.invoke(
        [
            ("system", system_prompt),
            (
                "user",
                "Preencha as quantidades da ListaMateriaisObra usando estes dados:\n"
                + json.dumps(user_payload, ensure_ascii=False, indent=2),
            ),
        ]
    )


lista_materiais_quantificada = estimar_quantidades_materiais(
    lista_materiais=lista_materiais_obra,
    dados_espaciais_ifc=dados_espaciais_ifc,
    build_digest_result=build_digest_result,
)
lista_materiais_quantificada


ListaMateriaisObra(obra=None, responsavel=None, data=None, moeda='BRL', observacoes='Lista gerada a partir de digest IFC2X3. Quantidades preenchidas apenas quando houve contagem direta, área IFC ou inferência técnica rastreável; preços e fornecedores não preenchidos nesta etapa.', areas=[AreaMateriaisObra(area='Fundacao', materiais=[MaterialObra(nome='Terra para aterro/plataforma', descricao='Terra/aterro identificado como camada de plataforma no pavimento de fundações.', quantidade=None, medida='m3', fornecedor='', lista_fornecedores=[], valor_unitario=None, valor_total=None, preco_a_vista=None, preco_a_prazo=None, num_parcelas=None, frete=None, perfil_produto=<PerfilProduto.MEDIO_CUSTO: 'Medio custo'>, origem=<OrigemMaterial.IFC: 'ifc'>, justificativa='Material citado diretamente no IFC em camada de plataforma com espessura 0,30 m, porém não há área/volume da plataforma para calcular m³.', nivel_confianca=85, referencias_ifc=['build_digest_result.materiais: Terra', 'IfcMaterialLayerS